Proyecto para deteccion de objetos por : 

-RECICLABLE---------------------------------------------------------------------------------------
Plásticos: Botellas de bebidas, envases de detergente, envases de alimentos (limpios).

Cartón: Cajas de cartón.

Vidrio: Botellas de bebidas y frascos.

Metales: Latas de aluminio (refrescos).

-NO RECICLABLE-------------------------------------------------------------------------------------
Unicel: Vasos, charolas, platos, empaques de electrodomésticos.

Plásticos de un solo uso: Bolsas, cubiertos desechables, popotes, envoltorios de golosinas.

Residuos sanitarios: Pañales, toallas sanitarias, papel de baño, cubrebocas, guantes de látex.

In [1]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D

In [2]:
# Rutas de imágenes
entrenamiento = 'CNN/Imagenes/TRAIN'
validacion = 'CNN/Imagenes/VALIDATION'

epocas          = 20
altura, anchura = 200, 200
batch_size      = 16
clases          = 2

# Generadores
entrenar = ImageDataGenerator(
    rescale=1/255,
    zoom_range=0.20,
    horizontal_flip=True,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1
)

validar = ImageDataGenerator(rescale=1/255)

imagenes_entrenamiento = entrenar.flow_from_directory(
    entrenamiento,
    target_size=(altura, anchura),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=True
)

imagenes_validacion = validar.flow_from_directory(
    validacion,
    target_size=(altura, anchura),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

#pasos
pasos            = imagenes_entrenamiento.samples // batch_size
pasos_validacion = imagenes_validacion.samples // batch_size

print("Clases:", imagenes_entrenamiento.class_indices)
print("Pasos entrenamiento:", pasos)
print("Pasos validación:", pasos_validacion)


Found 161 images belonging to 2 classes.
Found 35 images belonging to 2 classes.
Clases: {'N': 0, 'R': 1}
Pasos entrenamiento: 10
Pasos validación: 2


In [3]:
# MobileNetV2 ya fue entrenado con millones de imágenes
base = MobileNetV2(
    input_shape=(altura, anchura, 3),
    include_top=False,
    weights='imagenet'       # Pesos pre-entrenados
)
base.trainable = False       # No reentrenar la base

CNN = Sequential([
    base,
    GlobalAveragePooling2D(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(clases, activation='softmax')
])

C:\Users\catyr\AppData\Local\Temp\ipykernel_15504\2789768743.py:2: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base = MobileNetV2(


In [4]:
CNN.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["acc", "mse"]
)

historia = CNN.fit(
    imagenes_entrenamiento,
    validation_data=imagenes_validacion,
    epochs=epocas,
    steps_per_epoch=pasos,
    validation_steps=pasos_validacion,
    callbacks=[EarlyStopping(monitor='val_acc', patience=5, restore_best_weights=True)],
    verbose=1
)

# Ver si el modelo aprendió
print("Precisión final entrenamiento:", historia.history['acc'][-1])
print("Precisión final validación:", historia.history['val_acc'][-1])

Epoch 1/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 7s 301ms/step - acc: 0.7241 - loss: 0.6504 - mse: 0.2074 - val_acc: 0.9688 - val_loss: 0.1867 - val_mse: 0.0504
Epoch 2/20
 1/10 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step - acc: 0.7500 - loss: 0.5577 - mse: 0.1813

c:\Users\catyr\OneDrive\Desktop\ReciclaAI\CNN\Lib\site-packages\keras\src\trainers\epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - acc: 0.7500 - loss: 0.5577 - mse: 0.1813 - val_acc: 0.9688 - val_loss: 0.1690 - val_mse: 0.0451
Epoch 3/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 176ms/step - acc: 0.9034 - loss: 0.2322 - mse: 0.0711 - val_acc: 0.9062 - val_loss: 0.4764 - val_mse: 0.0970
Epoch 4/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - acc: 1.0000 - loss: 0.0738 - mse: 0.0154 - val_acc: 0.9062 - val_loss: 0.5140 - val_mse: 0.1001
Epoch 5/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 182ms/step - acc: 0.9586 - loss: 0.0980 - mse: 0.0321 - val_acc: 0.9375 - val_loss: 0.1432 - val_mse: 0.0476
Epoch 6/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - acc: 0.8750 - loss: 0.3634 - mse: 0.0808 - val_acc: 0.9375 - val_loss: 0.1321 - val_mse: 0.0448
Precisión final entrenamiento: 0.875
Precisión final validación: 0.9375


In [5]:
import os
os.makedirs("CNN/Imagenes/Modelo", exist_ok=True)
CNN.save("CNN/Imagenes/Modelo/cnn.keras")
print("Modelo guardado")

Modelo guardado


In [6]:
import cv2
import numpy as np
from tensorflow.keras.utils import img_to_array
from keras.models import load_model
import os

ruta_modelo = "CNN/Imagenes/Modelo"
print("Archivos en la carpeta:", os.listdir(ruta_modelo))

# Cargar según el formato que exista
if os.path.exists(f"{ruta_modelo}/cnn.keras"):
    cnn = load_model(f"{ruta_modelo}/cnn.keras")
    print("Modelo .keras cargado")
elif os.path.exists(f"{ruta_modelo}/cnn.h5"):
    cnn = load_model(f"{ruta_modelo}/cnn.h5")
    print("Modelo .h5 cargado")
else:
    print("No se encontró ningún modelo, entrena primero")
    exit()

altura, anchura = 200, 200

capture = cv2.VideoCapture(0)

while True:
    ret, frame = capture.read()

    if not ret:
        print("Error al leer la cámara")
        break

    # Clasificar cada frame
    evaluar_imagen = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    evaluar_imagen = cv2.resize(evaluar_imagen, (anchura, altura))
    evaluar_imagen = img_to_array(evaluar_imagen)
    evaluar_imagen = evaluar_imagen / 255.0
    evaluar_imagen = np.expand_dims(evaluar_imagen, axis=0)

    clase = cnn.predict(evaluar_imagen, verbose=0)
    arg_max = np.argmax(clase[0])

    if arg_max == 0:
        resultado = "No reciclable"
        color = (0, 0, 255)    # Rojo
    else:
        resultado = "Reciclable"
        color = (0, 255, 0)    # Verde

    # Mostrar resultado en la cámara
    cv2.putText(frame, resultado,
                (20, 50),
                cv2.FONT_HERSHEY_SIMPLEX,
                1.5,
                color,
                3)

    # Mostrar probabilidades
    prob = f"N:{clase[0][0]:.2f}  R:{clase[0][1]:.2f}"
    cv2.putText(frame, prob,
                (20, 100),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (255, 255, 255),
                2)

    cv2.imshow("Camara", frame)

    k = cv2.waitKey(5)
    if k == 27:    # ESC para salir
        break

capture.release()
cv2.destroyAllWindows()

Archivos en la carpeta: ['cnn.keras']
Modelo .keras cargado
